# Step 12 – End-to-end Demo (Round 2): সিলেটি কথা → বাংলা / English লেখা ও কণ্ঠ
**Runtime → Change runtime type → GPU (T4 যথেষ্ট)**, তারপর **Run all**।

- Pipeline: **Whisper-large-v3-turbo, Round 2** (`whisper-large-v3-turbo-round2/final`) → **NLLB-200 (fine-tuned)** → **MMS-TTS**। Round 2 মডেল না পেলে Round 1 মডেল (`-4spk/final`) নেবে, আর সেটা লিখে দেবে।
- কোনো training নেই, Drive থেকে শুধু পড়ে; Drive-এ কিছু লেখে না, কিছু মোছে না।
- প্রথমবার library ইনস্টল হলে session নিজে restart হবে ("session crashed") — তখন আবার **Run all**।
- শেষ cell-এ একটা **public লিংক** (`https://....gradio.live`) আসবে: ব্রাউজারে বা ফোনে খুলে মাইকে সিলেটি বলুন। Thesis-এর **Figure 5.5**-এর জন্য ওই পেজের একটা screenshot নিন।
- লিংক Colab বন্ধ করলে বন্ধ হয়ে যায়; defense-এর আগে আবার চালিয়ে নতুন লিংক নিন।

## 1. Library

In [ ]:
import os, sys, subprocess, time
from importlib.metadata import version, PackageNotFoundError
def _ver(p):
    try:
        return version(p)
    except PackageNotFoundError:
        return None
need = _ver("transformers") != "4.48.3" or _ver("gradio") is None or _ver("uroman") is None
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece", "uroman", "soundfile",
                    "transformers==4.48.3", "gradio>=5,<6", "huggingface_hub<1.0"], check=True)
    print("installed -> restarting the session; press Run all again")
    os.kill(os.getpid(), 9)
import torch
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(torch.cuda.get_device_name(0) if DEV == "cuda" else "GPU নেই -> CPU-তে চলবে (ধীর)",
      "| transformers", version("transformers"), "| gradio", version("gradio"))

## 2. Drive আর মডেলের ঠিকানা

In [ ]:
from google.colab import drive
if os.path.exists("/content/drive") and not os.path.ismount("/content/drive") and os.listdir("/content/drive"):
    new = "/content/drive_local_leftover_" + time.strftime("%H%M%S")
    os.rename("/content/drive", new); print("leftover local folder moved to", new)   # কিছু মোছা হয় না
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/Sylheti_Thesis"
def first(*paths):
    for p in paths:
        if os.path.exists(p + "/config.json"):
            return p
    return None
ASR_MODEL = first(BASE + "/sylheti_asr/whisper-large-v3-turbo-round2/final",
                  BASE + "/sylheti_asr/whisper-large-v3-turbo-4spk/final")
MT_MODEL = first(BASE + "/sylheti_mt/nllb-sylheti/final")
assert ASR_MODEL, "turbo final মডেল পাওয়া যায়নি (sylheti_asr/whisper-large-v3-turbo-round2/final)"
assert MT_MODEL, "NLLB final মডেল পাওয়া যায়নি (sylheti_mt/nllb-sylheti/final)"
ASR_NAME = "Whisper-large-v3-turbo, Round 2" if "round2" in ASR_MODEL else "Whisper-large-v3-turbo, Round 1 (Round 2 মডেল পাওয়া যায়নি)"
print("ASR:", ASR_MODEL, "->", ASR_NAME)
print("MT :", MT_MODEL)

## 3. মডেল লোড (৩–৫ মিনিট)

In [ ]:
import re, unicodedata, numpy as np
from transformers import (WhisperProcessor, WhisperForConditionalGeneration,
                          AutoTokenizer, AutoModelForSeq2SeqLM, VitsModel)
DT = torch.float16 if DEV == "cuda" else torch.float32

# ASR: same decoding as the Step 10 test (Bengali token, transcribe, greedy, max 440 tokens)
asr_proc = WhisperProcessor.from_pretrained(ASR_MODEL, language="bengali", task="transcribe")
asr = WhisperForConditionalGeneration.from_pretrained(ASR_MODEL, torch_dtype=DT).to(DEV).eval()
asr.generation_config.language = "bengali"; asr.generation_config.task = "transcribe"
asr.generation_config.forced_decoder_ids = None; asr.config.forced_decoder_ids = None

# MT: same settings as Step 3b / Step 11 (source ben_Beng, 4 beams)
mt_tok = AutoTokenizer.from_pretrained(MT_MODEL, src_lang="ben_Beng")
mt = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL, torch_dtype=DT).to(DEV).eval()

# TTS: pretrained MMS (no training), same as Step 4
try:
    import uroman as ur; _uroman = ur.Uroman()
except Exception:
    _uroman = None
tts = {k: (AutoTokenizer.from_pretrained(n), VitsModel.from_pretrained(n).to(DEV).eval())
       for k, n in [("bn", "facebook/mms-tts-ben"), ("en", "facebook/mms-tts-eng")]}
print("all models ready")

## 4. Pipeline-এর ফাংশন (cleaning আর বাংলা normalisation: Step 3b / Step 4-এর মতোই)

In [ ]:
ZW = dict.fromkeys(map(ord, "\u200b\u200c\ufeff\u00ad\u2060"), None)
def clean(s):                                   # same as Step 3b (MT input)
    s = unicodedata.normalize("NFC", str(s)).translate(ZW)
    s = re.sub("\u200d(?!্য)", "", s)
    s = re.sub(r"[\"\'“”‘’(),.।?!:;\-/*…\[\]|]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

# ---- Bangla text normalisation before TTS (copied from Step4_TTS_normalized, Appendix E) ----
_ONES = ("শূন্য এক দুই তিন চার পাঁচ ছয় সাত আট নয় দশ এগারো বারো তেরো চোদ্দ পনেরো ষোলো সতেরো আঠারো উনিশ "
         "বিশ একুশ বাইশ তেইশ চব্বিশ পঁচিশ ছাব্বিশ সাতাশ আটাশ উনত্রিশ ত্রিশ একত্রিশ বত্রিশ তেত্রিশ চৌত্রিশ পঁয়ত্রিশ ছত্রিশ সাঁইত্রিশ আটত্রিশ উনচল্লিশ "
         "চল্লিশ একচল্লিশ বিয়াল্লিশ তেতাল্লিশ চুয়াল্লিশ পঁয়তাল্লিশ ছেচল্লিশ সাতচল্লিশ আটচল্লিশ উনপঞ্চাশ পঞ্চাশ একান্ন বাহান্ন তিপ্পান্ন চুয়ান্ন পঞ্চান্ন ছাপ্পান্ন সাতান্ন আটান্ন উনষাট "
         "ষাট একষট্টি বাষট্টি তেষট্টি চৌষট্টি পঁয়ষট্টি ছেষট্টি সাতষট্টি আটষট্টি উনসত্তর সত্তর একাত্তর বাহাত্তর তিয়াত্তর চুয়াত্তর পঁচাত্তর ছিয়াত্তর সাতাত্তর আটাত্তর উনআশি "
         "আশি একাশি বিরাশি তিরাশি চুরাশি পঁচাশি ছিয়াশি সাতাশি আটাশি উননব্বই নব্বই একানব্বই বিরানব্বই তিরানব্বই চুরানব্বই পঁচানব্বই ছিয়ানব্বই সাতানব্বই আটানব্বই নিরানব্বই").split()
assert len(_ONES) == 100
_BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
def num_to_bn(n):
    if n < 100: return _ONES[n]
    parts = []
    for div, word in ((10**7, "কোটি"), (10**5, "লাখ"), (1000, "হাজার")):
        if n >= div:
            parts.append(num_to_bn(n // div) + " " + word); n %= div
    if n >= 100:
        parts.append(_ONES[n // 100] + "শো"); n %= 100
    if n: parts.append(_ONES[n])
    return " ".join(parts)
def _number(m):
    s = m.group(0).translate(_BN_DIGITS).replace(",", "")
    if "." in s:
        a, b = s.split(".", 1)
        return num_to_bn(int(a or 0)) + " দশমিক " + " ".join(_ONES[int(d)] for d in b)
    return num_to_bn(int(s))
def normalize_bn(text):
    t = unicodedata.normalize("NFC", str(text))
    t = t.replace("\u200c", "").replace("\u200d", "")
    t = re.sub(r"[০-৯0-9]+(?:[,][০-৯0-9]{2,3})*(?:\.[০-৯0-9]+)?", _number, t)
    t = t.replace("%", " শতাংশ").replace("&", " এবং ")
    t = re.sub(r"[/|\\]", " অথবা ", t)
    t = re.sub(r"[\"'“”‘’«»()\[\]{}<>*_#~^]", " ", t)
    t = re.sub(r"\s*[-–—]\s*", " ", t)
    t = re.sub(r"\.{2,}|…", "।", t)
    t = re.sub(r"(?<=[\u0980-\u09FF])\.(?=\s|$)", "।", t)
    t = re.sub(r"\s*([।?!,;:])\s*", r"\1 ", t)
    t = re.sub(r"([।?!])[।?!\s]*", r"\1 ", t)
    t = re.sub(r"\s+", " ", t).strip()
    if t and t[-1] not in "।?!": t += "।"
    return t

def load_any(path):
    """any audio/video file (m4a, mp4, 3gp, amr, wav, mp3 ...) -> 16 kHz mono float32 via ffmpeg"""
    raw = subprocess.run(["ffmpeg", "-nostdin", "-loglevel", "error", "-i", path, "-vn", "-ac", "1", "-ar", "16000",
                          "-f", "f32le", "-"], capture_output=True, timeout=120).stdout
    return 16000, np.frombuffer(raw, dtype=np.float32).copy()

def to_16k_mono(sr, a):
    a = np.asarray(a)
    if a.dtype.kind in "iu":
        a = a.astype(np.float32) / np.iinfo(a.dtype).max
    a = a.astype(np.float32)
    if a.ndim > 1: a = a.mean(axis=1)
    if sr != 16000:
        import librosa
        a = librosa.resample(a, orig_sr=sr, target_sr=16000)
    return a[: 30 * 16000]                      # the model was trained on clips up to 30 s

@torch.no_grad()
def transcribe(audio):
    sr, a = audio
    feats = asr_proc.feature_extractor(to_16k_mono(sr, a), sampling_rate=16000, return_tensors="pt").input_features
    ids = asr.generate(feats.to(DEV, DT), max_new_tokens=440)
    return asr_proc.tokenizer.batch_decode(ids, skip_special_tokens=True)[0].strip()

@torch.no_grad()
def translate(text, code_):
    enc = mt_tok([text or "-"], return_tensors="pt", truncation=True, max_length=256).to(DEV)
    n_new = min(256, enc["input_ids"].shape[1] * 2 + 10)
    ids = mt.generate(**enc, forced_bos_token_id=mt_tok.convert_tokens_to_ids(code_), num_beams=4, max_new_tokens=n_new)
    return mt_tok.batch_decode(ids, skip_special_tokens=True)[0].strip()

@torch.no_grad()
def speak(text, lang):
    text = (text or "").strip()
    if not text:
        return None
    text = normalize_bn(text) if lang == "bn" else text.lower()
    tok_, model_ = tts[lang]
    if getattr(tok_, "is_uroman", False) and _uroman is not None:
        text = _uroman.romanize_string(text)
    torch.manual_seed(0)                       # same text -> same voice every time
    wav = model_(**tok_(text, return_tensors="pt").to(DEV)).waveform[0].float().cpu().numpy()
    return (model_.config.sampling_rate, wav)

# quick self-test with typed text (no microphone needed)
_t = time.time()
s = "তুমি অনে আমার সামনে থাকি যাও"
print("bn:", translate(clean(s), "ben_Beng")); print("en:", translate(clean(s), "eng_Latn"))
print("tts ok:", speak("এটি একটি পরীক্ষা।", "bn") is not None, "| %.1f s" % (time.time() - _t))

In [ ]:
# Cell 4b: V1 voice-clone model (attempt 3, step 2,000, noise 0.3 — same as the MOS test)
V1_MODEL = BASE + "/voice_clone/model_V1_v3/best"
V1_NOISE = 0.3
if os.path.exists(V1_MODEL + "/config.json"):
    v1_tok = AutoTokenizer.from_pretrained(V1_MODEL)
    v1 = VitsModel.from_pretrained(V1_MODEL).to(DEV).eval()
    v1.noise_scale = V1_NOISE; v1.noise_scale_duration = V1_NOISE
    print("V1 voice model ready:", V1_MODEL)
else:
    v1 = None
    print("V1 model পাওয়া যায়নি:", V1_MODEL)

@torch.no_grad()
def speak_v1(text):
    text = (text or "").strip()
    if v1 is None or not text:
        return None
    text = normalize_bn(text)
    if getattr(v1_tok, "is_uroman", False) and _uroman is not None:
        text = _uroman.romanize_string(text)
    torch.manual_seed(0)                       # same text -> same voice every time
    wav = v1(**v1_tok(text, return_tensors="pt").to(DEV)).waveform[0].float().cpu().numpy()
    return (v1.config.sampling_rate, wav)

print("V1 test ok:", speak_v1("আমি এখন বাড়ি যাব।") is not None)

## 5. Demo (Gradio)
শেষে `Running on public URL: https://....gradio.live` লিংকটা খুলুন। Thesis-এর **Figure 5.5**-এর জন্য এই পেজের screenshot নিন।

In [ ]:
import gradio as gr

def run_text(syl, timing=None):
    timing = timing or {}
    t = time.time(); syl_c = clean(syl)
    bn = translate(syl_c, "ben_Beng"); en = translate(syl_c, "eng_Latn"); timing["MT"] = time.time() - t
    t = time.time(); a_bn = speak(bn, "bn"); a_en = speak(en, "en"); timing["TTS"] = time.time() - t
    t = time.time(); v_bn = speak_v1(bn); v_syl = speak_v1(syl); timing["V1 voice"] = time.time() - t
    info = " | ".join(f"{k}: {v:.2f} s" for k, v in timing.items()) + f" | total: {sum(timing.values()):.2f} s"
    return syl, bn, en, a_bn, a_en, v_bn, v_syl, info

EMPTY = ("", "", "", None, None, None, None)
def run_speech(audio):
    if audio is None:
        return EMPTY + ("আগে অডিও রেকর্ড বা আপলোড করুন",)
    t = time.time(); syl = transcribe(audio); timing = {"ASR": time.time() - t}
    return run_text(syl, timing)

def run_file(f):
    if f is None:
        return EMPTY + ("আগে একটা ফাইল দিন",)
    audio = load_any(f if isinstance(f, str) else f.name)
    if len(audio[1]) == 0:
        return EMPTY + ("ফাইলটা পড়া যায়নি — অডিও আছে কিনা দেখুন",)
    return run_speech(audio)

def run_typed(syl):
    if not (syl or "").strip():
        return EMPTY + ("সিলেটি বাক্য লিখুন",)
    return run_text(syl)

def respeak(syl, bn, en):
    t = time.time()
    out = (speak(bn, "bn"), speak(en, "en"), speak_v1(bn), speak_v1(syl))
    return out + (f"TTS (edited text): {time.time() - t:.2f} s",)

with gr.Blocks(title="Sylheti Speech Translation") as demo:
    gr.Markdown("## সিলেটি → বাংলা / English Speech-to-Speech Translation\n"
                f"{ASR_NAME} (fine-tuned) → NLLB-200-distilled-600M (fine-tuned) → MMS-TTS"
                + (" + V1 voice (MMS-TTS adapted to one speaker)" if v1 is not None else ""))
    with gr.Tabs():
        with gr.Tab("🎤 সিলেটিতে বলুন"):
            mic = gr.Audio(sources=["microphone"], type="numpy", label="মাইকে সিলেটি বলুন (৩০ সেকেন্ডের কম)")
            b1 = gr.Button("Translate", variant="primary")
        with gr.Tab("📁 অডিও ফাইল"):
            upl = gr.File(label="m4a / mp4 / 3gp / amr / wav / mp3 ...", type="filepath")
            b4 = gr.Button("Translate", variant="primary")
        with gr.Tab("⌨️ সিলেটি লিখুন"):
            typed = gr.Textbox(label="সিলেটি বাক্য", lines=2)
            b2 = gr.Button("Translate", variant="primary")
    syl_out = gr.Textbox(label="সিলেটি লেখা (ASR, দরকার হলে ঠিক করুন)", interactive=True)
    with gr.Row():
        bn_out = gr.Textbox(label="বাংলা অনুবাদ (দরকার হলে ঠিক করে নিচের বোতাম চাপুন)", interactive=True)
        en_out = gr.Textbox(label="English translation (editable)", interactive=True)
    gr.Markdown("**সাধারণ কণ্ঠ (MMS-TTS)**")
    with gr.Row():
        bn_audio = gr.Audio(label="বাংলা কণ্ঠ", type="numpy")
        en_audio = gr.Audio(label="English speech", type="numpy")
    gr.Markdown("**V1-এর নিজের কণ্ঠে (voice clone)**")
    with gr.Row():
        v1_bn_audio = gr.Audio(label="বাংলা অনুবাদ — V1-এর কণ্ঠে", type="numpy")
        v1_syl_audio = gr.Audio(label="সিলেটি লেখা — V1-এর কণ্ঠে", type="numpy")
    b3 = gr.Button("🔊 ঠিক করা লেখা থেকে আবার কণ্ঠ বানান")
    info = gr.Markdown()
    outs = [syl_out, bn_out, en_out, bn_audio, en_audio, v1_bn_audio, v1_syl_audio, info]
    b1.click(run_speech, mic, outs)
    b2.click(run_typed, typed, outs)
    b4.click(run_file, upl, outs)
    b3.click(respeak, [syl_out, bn_out, en_out], [bn_audio, en_audio, v1_bn_audio, v1_syl_audio, info])

demo.queue().launch(share=True, debug=True)

---
**নোট (thesis-এ লেখার জন্য)**
- মডেল শুধু পড়া (read) সিলেটি বাক্যে train করা; স্বাভাবিক কথোপকথনে ভুল বেশি হবে (Section 6.6)।
- বাংলা TTS-এ ইংরেজি অক্ষর থাকলে MMS পড়তে পারে না; তখন বাংলা বক্সে ঠিক করে "আবার কণ্ঠ বানান" চাপুন।
- থামাতে সেলের ⏹ বোতাম চাপুন।